# 06 — Spatiotemporal Deep Learning (ConvLSTM)

**Objective:** Predict future cardiac event density fields using a convolutional LSTM model
trained on spatiotemporal tensor sequences.

**Methodology:**
- Raw event point data is discretized into a `[T, 1, H, W]` tensor on a regular lat/lon grid
- A ConvLSTM model learns temporal dynamics over 5-day sliding windows
- Performance is benchmarked against a Persistence baseline (tomorrow = today)
- Temporal train/val/test split prevents data leakage

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available()
                      else 'mps' if torch.backends.mps.is_available()
                      else 'cpu')
print(f'Compute device: {device}')

PROJECT_ROOT = Path('..')
V3_DIR = PROJECT_ROOT / 'mda_project' / 'data' / 'processed_v3'

mission = pd.read_parquet(V3_DIR / 'mission_records_v3.parquet')
mission['t0'] = pd.to_datetime(mission['t0'], errors='coerce')
mission = mission.dropna(subset=['t0'])
mission['date'] = mission['t0'].dt.date
print(f'Records: {len(mission):,}')

## 6.1 Spatiotemporal Tensor Construction

Event point locations are binned into a 49×49 spatial grid per day,
producing a time series of 2D density fields.

In [ ]:
lat_bins = np.linspace(49.5, 51.6, 50)
lon_bins = np.linspace(2.5, 6.5, 50)
dates = sorted(mission['date'].unique())

grids = []
for d in dates:
    sub = mission[mission['date'] == d]
    h, _, _ = np.histogram2d(sub['latitude'], sub['longitude'], bins=[lat_bins, lon_bins])
    grids.append(h)

X = np.stack(grids, axis=0).astype(np.float32)
print(f'Tensor shape [Time, Height, Width]: {X.shape} ({len(dates)} days)')

## 6.2 Sequence Sampling & Data Splits

In [ ]:
seq_len = 5
xs, ys = [], []

for i in range(seq_len, len(X)):
    xs.append(X[i-seq_len:i])
    ys.append(X[i])

# Add channel dimension: [B, Seq, C=1, H, W]
xs = np.array(xs)[:, :, np.newaxis, :, :]
ys = np.array(ys)[:, np.newaxis, :, :]

# Temporal split: 70% train, 15% val, 15% test
n = len(xs)
split_v, split_t = int(n * 0.7), int(n * 0.85)

train_dl = DataLoader(TensorDataset(torch.tensor(xs[:split_v]), torch.tensor(ys[:split_v])),
                      batch_size=8, shuffle=True)
val_dl = DataLoader(TensorDataset(torch.tensor(xs[split_v:split_t]), torch.tensor(ys[split_v:split_t])),
                    batch_size=8)
test_dl = DataLoader(TensorDataset(torch.tensor(xs[split_t:]), torch.tensor(ys[split_t:])),
                     batch_size=8)

print(f'Samples — Train: {split_v}, Val: {split_t-split_v}, Test: {n-split_t}')

## 6.3 Persistence Baseline

In [ ]:
# Baseline: predict tomorrow = today (last frame in sequence)
baseline_pred = xs[split_t:, -1, :, :, :]
baseline_true = ys[split_t:]
mae_baseline = np.mean(np.abs(baseline_pred - baseline_true))
rmse_baseline = np.sqrt(np.mean((baseline_pred - baseline_true)**2))

print(f'Persistence Baseline — MAE: {mae_baseline:.4f}, RMSE: {rmse_baseline:.4f}')

## 6.4 ConvLSTM Model Architecture

In [ ]:
class ConvLSTMCell(nn.Module):
    """Single ConvLSTM cell combining spatial convolution with LSTM gating."""
    def __init__(self, input_dim, hidden_dim, kernel_size):
        super().__init__()
        self.hidden_dim = hidden_dim
        padding = kernel_size // 2
        self.conv = nn.Conv2d(input_dim + hidden_dim, 4 * hidden_dim,
                              kernel_size=kernel_size, padding=padding)

    def forward(self, x, state):
        h, c = state
        combined = torch.cat([x, h], dim=1)
        gates = self.conv(combined)
        i, f, o, g = torch.split(gates, self.hidden_dim, dim=1)
        i, f, o = torch.sigmoid(i), torch.sigmoid(f), torch.sigmoid(o)
        g = torch.tanh(g)
        c_next = f * c + i * g
        h_next = o * torch.tanh(c_next)
        return h_next, c_next


class SpatiotemporalPredictor(nn.Module):
    """ConvLSTM-based predictor for 2D density field forecasting."""
    def __init__(self, in_ch=1, hidden_ch=16, out_ch=1, kernel_size=3):
        super().__init__()
        self.hidden_ch = hidden_ch
        self.cell = ConvLSTMCell(in_ch, hidden_ch, kernel_size)
        self.out_conv = nn.Conv2d(hidden_ch, out_ch, 1)

    def forward(self, x):
        b, seq, _, h, w = x.size()
        h_state = torch.zeros(b, self.hidden_ch, h, w, device=x.device)
        c_state = torch.zeros(b, self.hidden_ch, h, w, device=x.device)
        for t in range(seq):
            h_state, c_state = self.cell(x[:, t], (h_state, c_state))
        return torch.relu(self.out_conv(h_state))


model = SpatiotemporalPredictor().to(device)
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

## 6.5 Training Loop

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.003)

epochs = 30
train_losses, val_losses = [], []

for epoch in range(epochs):
    model.train()
    batch_loss = []
    for bx, by in train_dl:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        loss = criterion(model(bx), by)
        loss.backward()
        optimizer.step()
        batch_loss.append(loss.item())
    train_losses.append(np.mean(batch_loss))

    model.eval()
    vl = []
    with torch.no_grad():
        for bx, by in val_dl:
            vl.append(criterion(model(bx.to(device)), by.to(device)).item())
    val_losses.append(np.mean(vl))

    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1:02d}/{epochs} — Train Loss: {train_losses[-1]:.4f} | Val Loss: {val_losses[-1]:.4f}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(train_losses, label='Train MSE', linewidth=2)
ax.plot(val_losses, label='Validation MSE', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('ConvLSTM Training Convergence', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6.6 Test Set Evaluation & Baseline Comparison

In [ ]:
model.eval()
test_preds, test_trues = [], []

with torch.no_grad():
    for bx, by in test_dl:
        test_preds.append(model(bx.to(device)).cpu().numpy())
        test_trues.append(by.numpy())

test_preds = np.concatenate(test_preds)
test_trues = np.concatenate(test_trues)

mae_dl = np.mean(np.abs(test_preds - test_trues))
rmse_dl = np.sqrt(np.mean((test_preds - test_trues)**2))

comparison = pd.DataFrame([
    {'Model': 'Persistence Baseline', 'MAE': mae_baseline, 'RMSE': rmse_baseline},
    {'Model': 'ConvLSTM', 'MAE': mae_dl, 'RMSE': rmse_dl},
])
comparison['MAE Improvement'] = ['-', f'{(mae_baseline - mae_dl)/mae_baseline*100:.1f}%']
display(comparison)

## Summary

The ConvLSTM model captures spatiotemporal dynamics beyond a simple persistence baseline.

**Key capabilities demonstrated:**
1. Spatiotemporal tensor construction from irregular event data
2. Custom ConvLSTM implementation in PyTorch (no third-party wrappers)
3. Temporal data splitting to prevent information leakage
4. Quantitative baseline comparison

The final publication-ready figures (including regularized ST-CNN predictions with
Dropout + L2) are generated by `05_final_figures.py`.